In [ ]:
# %% 최근 파일 5개 삭제
import os, glob
from datetime import datetime

OUTPUT_DIR = "./data/7_qwen_test"

files = glob.glob(os.path.join(OUTPUT_DIR, "**", "*.json"), recursive=True)
files_with_mtime = [(f, os.path.getmtime(f)) for f in files]
files_with_mtime.sort(key=lambda x: x[1], reverse=True)

for f, mtime in files_with_mtime[:7]:
    ts = datetime.fromtimestamp(mtime).strftime("%m-%d %H:%M:%S")
    rel = os.path.relpath(f, OUTPUT_DIR)
    os.remove(f)
    print(f"  삭제: {ts} {rel}")

print(f"\n✅ {len(files_with_mtime)}개 삭제 완료")

In [ ]:
"""
동시 텔롭 최대 개수 분석
- 6_telop_instances의 JSON에서 (start_sec, end_sec) 기반으로
  0.1초 bin별 동시 활성 인스턴스 수를 계산
"""
import os
import json
import glob
import numpy as np
from collections import Counter
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

INST_DIR = "./data/6_telop_instances"
BIN_SIZE = 0.1
NUM_WORKERS = 32


def analyze_one(json_path: str) -> dict:
    """영상 1개의 동시 텔롭 통계"""
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    instances = data.get("instances", [])
    if not instances:
        return {"max_simul": 0, "mean_simul": 0.0, "n_instances": 0, "path": json_path}

    # bin별 동시 활성 인스턴스 수
    bins = {}
    for inst in instances:
        s, e = inst["start_sec"], inst["end_sec"]
        b_start = int(s / BIN_SIZE)
        b_end = int(e / BIN_SIZE)
        for b in range(b_start, b_end):
            bins[b] = bins.get(b, 0) + 1

    if not bins:
        return {"max_simul": 0, "mean_simul": 0.0, "n_instances": len(instances), "path": json_path}

    counts = list(bins.values())
    return {
        "max_simul": max(counts),
        "mean_simul": np.mean(counts),
        "n_instances": len(instances),
        "path": json_path,
    }


def main():
    json_paths = sorted(glob.glob(os.path.join(INST_DIR, "**", "*.json"), recursive=True))
    print(f"📁 분석 대상: {len(json_paths):,}개 영상")

    results = []
    with ProcessPoolExecutor(max_workers=NUM_WORKERS) as pool:
        futures = {pool.submit(analyze_one, p): p for p in json_paths}
        for fut in tqdm(as_completed(futures), total=len(futures), desc="분석"):
            results.append(fut.result())

    # ── 전체 통계 ──
    max_simuls = np.array([r["max_simul"] for r in results])
    mean_simuls = np.array([r["mean_simul"] for r in results])

    print(f"\n{'='*60}")
    print(f"📊 동시 텔롭 개수 통계 ({len(results):,}개 영상)")
    print(f"{'='*60}")

    print(f"\n  영상별 최대 동시 텔롭 수:")
    print(f"    mean:   {max_simuls.mean():.1f}")
    print(f"    median: {np.median(max_simuls):.1f}")
    print(f"    p90:    {np.percentile(max_simuls, 90):.0f}")
    print(f"    p95:    {np.percentile(max_simuls, 95):.0f}")
    print(f"    p99:    {np.percentile(max_simuls, 99):.0f}")
    print(f"    max:    {max_simuls.max():.0f}")

    print(f"\n  영상별 평균 동시 텔롭 수:")
    nonzero_means = mean_simuls[mean_simuls > 0]
    print(f"    mean:   {nonzero_means.mean():.2f}")
    print(f"    median: {np.median(nonzero_means):.2f}")

    # ── 분포 ──
    print(f"\n  최대 동시 텔롭 수 분포:")
    print(f"    {'구간':<12} {'영상수':>8} {'비율':>8} {'누적':>8}")
    print(f"    {'─'*12} {'─'*8} {'─'*8} {'─'*8}")
    bins_edges = [0, 1, 2, 3, 4, 5, 6, 8, 10, 15, 20, 50, 1000]
    cumul = 0
    for i in range(len(bins_edges) - 1):
        lo, hi = bins_edges[i], bins_edges[i + 1]
        cnt = int(((max_simuls >= lo) & (max_simuls < hi)).sum())
        cumul += cnt
        label = f"{lo}" if hi - lo == 1 else f"{lo}~{hi-1}"
        print(f"    {label:<12} {cnt:>8} {cnt/len(max_simuls)*100:>7.1f}% {cumul/len(max_simuls)*100:>7.1f}%")

    # ── top 10 ──
    print(f"\n  동시 텔롭 최대인 영상 top 10:")
    top10 = sorted(results, key=lambda r: r["max_simul"], reverse=True)[:10]
    for r in top10:
        rel = os.path.relpath(r["path"], INST_DIR)
        print(f"    max={r['max_simul']:>3}  inst={r['n_instances']:>4}  {rel}")

    # ── 채널별 평균 ──
    channel_maxs = {}
    for r in results:
        ch = r["path"].split(os.sep)[-2] if os.sep in r["path"] else "unknown"
        if ch not in channel_maxs:
            channel_maxs[ch] = []
        channel_maxs[ch].append(r["max_simul"])

    ch_avg = {ch: np.mean(vs) for ch, vs in channel_maxs.items()}
    print(f"\n  채널별 평균 최대 동시 텔롭 top 10:")
    for ch, avg in sorted(ch_avg.items(), key=lambda x: x[1], reverse=True)[:10]:
        print(f"    avg_max={avg:>5.1f}  ({len(channel_maxs[ch]):>3}개 영상)  {ch}")

    # ── K값 제안 ──
    print(f"\n{'='*60}")
    print(f"💡 K값 제안")
    print(f"{'='*60}")
    for p in [90, 95, 99]:
        v = np.percentile(max_simuls, p)
        cover = (max_simuls <= v).sum() / len(max_simuls) * 100
        print(f"  K={int(v):>2} → {cover:.1f}% 영상 커버 (p{p})")


if __name__ == "__main__":
    main()

In [ ]:
# %% 텔롭 인스턴스 수 분포 확인
import os
import numpy as np
import polars as pl
from tqdm.auto import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

INST_DIR = "./data/6_telop_instances"


def analyze_one(args):
    channel, fname = args
    path = os.path.join(INST_DIR, channel, fname)
    try:
        df = pl.read_parquet(path, glob=False)
    except Exception:
        return None

    n_total = len(df)
    if n_total == 0:
        return {"n_total": 0, "max_concurrent": 0, "channel": channel}

    # 같은 시점 최대 공존 개수 (sweep line)
    events = []
    for row in df.iter_rows(named=True):
        events.append((float(row["start_sec"]), 1))
        events.append((float(row["end_sec"]), -1))
    events.sort(key=lambda x: (x[0], -x[1]))  # 같은 시각이면 시작이 먼저

    cur = 0
    mx = 0
    for _, d in events:
        cur += d
        mx = max(mx, cur)

    return {"n_total": n_total, "max_concurrent": mx, "channel": channel}


# 태스크 수집
tasks = []
for channel in sorted(os.listdir(INST_DIR)):
    ch_dir = os.path.join(INST_DIR, channel)
    if not os.path.isdir(ch_dir):
        continue
    for fname in os.listdir(ch_dir):
        if fname.endswith(".parquet"):
            tasks.append((channel, fname))

print(f"🎯 분석 대상: {len(tasks):,}개 영상")

# 병렬 실행
results = []
with ProcessPoolExecutor(max_workers=32) as pool:
    futures = [pool.submit(analyze_one, t) for t in tasks]
    for fut in tqdm(as_completed(futures), total=len(futures), desc="분석"):
        r = fut.result()
        if r is not None:
            results.append(r)

# 집계
totals = np.array([r["n_total"] for r in results])
concurrents = np.array([r["max_concurrent"] for r in results])

def summary(arr, name):
    print(f"\n📊 {name}")
    print(f"  mean:   {arr.mean():.1f}")
    print(f"  median: {np.median(arr):.0f}")
    print(f"  std:    {arr.std():.1f}")
    print(f"  min:    {arr.min()}")
    print(f"  max:    {arr.max()}")
    print(f"  p90:    {np.percentile(arr, 90):.0f}")
    print(f"  p95:    {np.percentile(arr, 95):.0f}")
    print(f"  p99:    {np.percentile(arr, 99):.0f}")
    print(f"  p99.9:  {np.percentile(arr, 99.9):.0f}")


summary(totals, "영상당 총 텔롭 인스턴스 수")
summary(concurrents, "영상당 동시 공존 최대 인스턴스 수")

# 구간별 분포
print(f"\n📊 영상당 총 인스턴스 수 구간 분포")
bins = [0, 10, 50, 100, 200, 500, 1000, 2000, 5000, 100000]
for lo, hi in zip(bins[:-1], bins[1:]):
    n = ((totals >= lo) & (totals < hi)).sum()
    pct = n / len(totals) * 100
    print(f"  {lo:>5}~{hi:<6}: {n:>6,} ({pct:>5.1f}%)")

print(f"\n📊 동시 공존 최대 개수 구간 분포")
bins_c = [0, 5, 10, 20, 50, 100, 200, 500, 10000]
for lo, hi in zip(bins_c[:-1], bins_c[1:]):
    n = ((concurrents >= lo) & (concurrents < hi)).sum()
    pct = n / len(concurrents) * 100
    print(f"  {lo:>4}~{hi:<6}: {n:>6,} ({pct:>5.1f}%)")

# 채널별 평균도 같이 보기 (어떤 채널이 텔롭을 많이 쓰는지)
from collections import defaultdict
by_ch = defaultdict(list)
for r in results:
    by_ch[r["channel"]].append(r["n_total"])

ch_means = [(ch, np.mean(v)) for ch, v in by_ch.items()]
ch_means.sort(key=lambda x: -x[1])
print(f"\n📊 영상 평균 인스턴스 수 top/bottom 10 채널")
print("  [TOP 10]")
for ch, m in ch_means[:10]:
    print(f"    {ch:40s} {m:7.1f}")
print("  [BOTTOM 10]")
for ch, m in ch_means[-10:]:
    print(f"    {ch:40s} {m:7.1f}")